# Sweep selectors against a saved oracle

This notebook evaluates Paper greedy and a family of tile selectors at the cases recorded in `oracle_reference.csv`. It reads the activation archive and two-line latency model, checks their hashes against the reference, then computes masks and ratio gaps. No oracle solve occurs here.

Run from this directory with NumPy, pandas, PyTorch and the native `vlmflash` dependencies installed. All experiment-specific code is below; the library supplies Paper's window selector. The last cell replaces `selector_results.csv`. Changing the tile grid requires rerunning only this notebook and the analysis.

The geometric branch connects the saturation-scale mask to individual-row selection. Whether finer selection improves the trade-off, or produces an abrupt deterioration, is an empirical question.

In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd

MODEL = Path("../01/latency_model.json")
TRACE = Path("activation_traces.npz")
REFERENCE = Path("oracle_reference.csv")
RESULTS = Path("selector_results.csv")
UPSTREAM = Path("../../vlm-flash")
PAPER_IMPL = "native"
COARSE_SCALES = [1, 1.25, 1.5, 1.75, 2]

## Validate the reference

In [2]:
input_hashes = {"trace_sha256": hashlib.sha256(TRACE.read_bytes()).hexdigest(),
                "model_sha256": hashlib.sha256(MODEL.read_bytes()).hexdigest()}
reference = pd.read_csv(REFERENCE)
assert not reference.empty and not reference.duplicated(["trace_id", "fraction"]).any()
for name, digest in input_hashes.items():
    assert reference[name].eq(digest).all(), f"{name} changed; regenerate the oracle reference."
assert (reference.eta_upper >= reference.eta_lower).all()
reference_sha256 = hashlib.sha256(REFERENCE.read_bytes()).hexdigest()

## Read the recorded cases

Each trace supplies a nonnegative activation-magnitude vector. Normalize it to mean one and use the saved requested count $R$; the reference determines which traces and budgets are evaluated. Float32 rounding matches the selector inputs used to construct the reference.

In [3]:
with np.load(TRACE, allow_pickle=False) as archive:
    manifest = json.loads(str(archive["manifest_json"].item()))
    traces = {t["trace_id"]: {**t, "values": archive[t["array"]].astype(float)}
              for t in manifest["traces"]}
weight_bytes = {"float16": 2, "bfloat16": 2, "float32": 4}[manifest["model_dtype"]]
assert set(reference.trace_id) <= traces.keys()

## Objective and cost model

Let $v_i\ge0$ be the importance of channel $i$, and let $\mathcal C(M)$ contain the maximal contiguous runs selected by a binary mask $M$. Define

$$I(M)=\sum_{i=0}^{N-1}v_iM_i,\qquad L(M)=\sum_{C\in\mathcal C(M)}T(|C|),\qquad
\eta_R^\star=\max_{\|M\|_1=R}\frac{I(M)}{L(M)}.$$

For a requested fraction $f$, set $R=\max(1,\lfloor fN\rfloor)\le N$. Adjacent selected intervals form one run and are charged once. The oracle assumes the continuous two-line cost

$$T(0)=0,\qquad T(\ell)=\begin{cases}a+b_1\ell,&1\le\ell\le s,\\b_2\ell,&\ell>s,\end{cases}
\qquad a=(b_2-b_1)s,\quad 0\le b_1\le b_2,\quad b_2>0.$$

The input model is $T_{\rm KiB}(z)=c_1z+\delta\max(z_{\rm 2L},z)$ in milliseconds. For a weight row of $w$ KiB, substituting $z=w\ell$ gives

$$s=\frac{z_{\rm 2L}}w,\qquad b_1=wc_1,\qquad b_2=w(c_1+\delta).$$

The boundary $s$ remains real-valued in the cost. Lengths outside the measured range use the model's linear extension. The coefficients below are the fixed input to this experiment.

In [4]:
model = json.loads(MODEL.read_text())
s_kib = model["s_kib"]
c1 = model["c1_ms_per_kib"]
delta = model["delta_ms_per_kib"]
s99 = model["s99_kib"]

In [5]:
def costs(n, s, b1, b2):
    length = np.arange(n + 1)
    T = b2 * length + (b2 - b1) * np.maximum(s - length, 0)
    T[0] = 0
    return T


def runs(mask):
    edges = np.flatnonzero(np.diff(np.r_[False, mask, False]))
    return edges.reshape(-1, 2)


def metrics(v, mask, T):
    spans = runs(mask)
    return v[mask].sum(), T[spans[:, 1] - spans[:, 0]].sum()

## Tile granularity

Let $s$ be the real-valued saturation boundary in rows. In addition to the coarse branch $\alpha\in\{1,1.25,1.5,1.75,2\}$, evaluate

$$B_j=\max(1,\lceil s/2^j\rceil),\qquad j=0,1,\ldots,\max(0,\lceil\log_2 s\rceil).$$

The last width is one row. Keep the first occurrence of each integer width, per trace; the coarse branch is considered first. `tile_scale` records the requested multiplier and `tile_width` the realized row count. Shapes can therefore have different method lists.

At each width, select the most important full tiles and a disjoint residual interval, comparing offsets zero and half a tile by importance per merged-run cost. At $B=1$, each full tile is a row and the residual is zero: this is Top-$R$. Ties prefer smaller channel indices. The endpoint appears once, labeled `Top-R`.

In [6]:
def tile_grid(s):
    levels = max(0, int(np.ceil(np.log2(max(1, s)))))
    scales = [(f"{a:g}s", float(a)) for a in COARSE_SCALES]
    scales += [(f"s/2^{j}", 2.**-j) for j in range(1, levels + 1)]
    seen, grid = set(), []
    for label, scale in scales:
        width = max(1, int(np.ceil(scale * s)))
        if width not in seen:
            name = "Top-R" if width == 1 else f"Tiles ({label})"
            grid.append((name, scale, width))
            seen.add(width)
    return grid

In [7]:
def top_r(v, R):
    mask = np.zeros(len(v), bool)
    mask[np.argsort(-v, kind="stable")[:R]] = True
    return mask

The residual interval must avoid all full tiles already selected.

In [8]:
def fill_residual(mask, P, remaining):
    if not remaining:
        return True
    occupied = np.r_[0, np.cumsum(mask)]
    valid = occupied[remaining:] == occupied[:-remaining]
    if not valid.any():
        return False
    start = np.argmax(np.where(valid, P[remaining:] - P[:-remaining], -np.inf))
    mask[start:start + remaining] = True
    return True

The two offsets compete using the same merged-run cost as the oracle. One-row tiles use the original importance values directly, avoiding prefix-subtraction roundoff in ties.

In [9]:
def tiles(v, R, T, B):
    P = np.r_[0., np.cumsum(v)]
    if R < B:
        start = np.argmax(P[R:] - P[:-R])
        mask = np.zeros(len(v), bool)
        mask[start:start + R] = True
        return mask
    choices = []
    k, remaining = divmod(R, B)
    for offset in dict.fromkeys((0, B // 2)):
        starts = np.arange(offset, len(v) - B + 1, B)
        if len(starts) < k:
            continue
        weights = v[starts] if B == 1 else P[starts + B] - P[starts]
        mask = np.zeros(len(v), bool)
        for start in starts[np.argsort(-weights, kind="stable")[:k]]:
            mask[start:start + B] = True
        if not fill_residual(mask, P, remaining):
            continue
        I, L = metrics(v, mask, T)
        choices.append((I / L, mask))
    return max(choices, key=lambda row: row[0])[1]

## Comparison selectors

**Paper greedy** ranks candidate windows by importance divided by their modeled cost, then accepts non-overlapping whole windows that fit the remaining budget. The native implementation supplies this selection. `PAPER_GEOMETRY` specifies window start/step and stride cap in KiB for each input-by-output shape. The ceiling is $z_{99}$, the first size reaching 99% of peak smoothed training throughput in the input profile.

Whole windows can leave the budget unfilled: $|M_P|\le R$. We keep that mask as returned. Final costs for every method use maximal merged runs through `metrics`.

In [10]:
PAPER_GEOMETRY = {"896x128": (8, 8), "896x896": (8, 8),
                  "896x4864": (8, 8), "4864x896": (12, 16)}


def load_paper():
    import sys, torch
    sys.path.insert(0, str((UPSTREAM / "src").resolve()))
    import vlmflash
    table = vlmflash.LatencyTable({
        z: c1*z + delta*max(s_kib, z)
        for z in range(1, int(np.ceil(max(s_kib, s99))) + 2)})

    def select(v, R, w, shape):
        start, jump = PAPER_GEOMETRY[shape]
        params = vlmflash.ChunkParams(start_kb=start, step_kb=start,
                                     jump_cap_kb=jump, end_kb=s99 + 1)
        result = vlmflash.select_chunks(torch.tensor(v, dtype=torch.float32),
            R, w, table, params, impl=PAPER_IMPL)
        return result.mask.cpu().numpy()

    return select

## Comparing ratios at the same budget

For a mask with exactly $R$ rows and ratio $u_M=I(M)/L(M)$, define $g(M)=100(1-u_M/\eta_R^\star)$. The oracle bracket gives

$$\max(0,100(1-u_M/\ell))\le g(M)\le\max(0,100(1-u_M/u)).$$

We report the upper end; the interval width is at most $100\epsilon$ percentage points in exact arithmetic. An underfilled Paper mask is outside the fixed-$R$ feasible set, so its gap is undefined. Empty selections and zero-importance cases also have no ratio gap, but retain their other measurements and status.

The CSV records the requested and selected counts, importance, latency and run intervals. Fragmentation is summarized by the number of runs and their total short-run deficit,

$$D(M)=\sum_{C\in\mathcal C(M)}(s-|C|)_+,\qquad
L(M)=b_2|M|+(b_2-b_1)D(M).$$

This identity also applies to underfilled masks. The analysis notebook reconstructs mask images from the saved intervals. The save cell writes the completed comparison to CSV.

In [11]:
def mask_stats(v, mask, T, s, penalty, R, lower, upper):
    selected = int(mask.sum())
    I, L = metrics(v, mask, T)
    spans = runs(mask)
    lengths = np.diff(spans, axis=1).ravel()
    shortfall = np.maximum(s - lengths, 0).sum()
    comparable = selected == R and lower > 0
    status = "empty_selection" if selected == 0 else "ok" if v.any() else "zero_importance"
    return dict(
        R_selected=selected, fill_pct=100*selected/R, status=status,
        importance=I, latency=L, retained_pct=100*I/v.sum() if v.any() else np.nan,
        chunks=len(spans), short_chunks=int((lengths < s).sum()),
        shortfall_rows=shortfall, overhead_ms=penalty*shortfall,
        gap_lower_pct=max(0, 100*(1-I/L/lower)) if comparable else np.nan,
        gap_upper_pct=max(0, 100*(1-I/L/upper)) if comparable else np.nan,
        intervals=json.dumps(spans.tolist()))

## Prepare one trace

Convert the common KiB cost model to row costs once. Validate the trace shape and requested counts against the reference before selection.

In [12]:
def prepare(trace, rows):
    v = trace["values"]
    assert np.isfinite(v).all() and (v >= 0).all()
    v = (v / v.mean()).astype(np.float32).astype(float) if v.any() else v
    assert rows.N.eq(len(v)).all() and rows["shape"].eq(trace["shape"]).all()
    assert rows.R_nominal.between(1, len(v)).all()
    w = weight_bytes * trace["d"] / 1024
    s, b1, b2 = s_kib / w, w*c1, w*(c1+delta)
    return v, w, s, b2-b1, costs(len(v), s, b1, b2)

For each case, compute Paper once and each distinct tile width once. The one-row endpoint is checked against stable Top-$R$.

In [13]:
def selected_masks(v, R, w, s, T, shape):
    yield "Paper greedy", np.nan, np.nan, paper(v, R, w, shape)
    for name, scale, width in tile_grid(s):
        mask = tiles(v, R, T, width)
        assert int(mask.sum()) == R
        if width == 1:
            assert np.array_equal(mask, top_r(v, R))
        yield name, scale, width, mask

## Run the sweep

In [14]:
paper = load_paper()
records = []
for trace_id, rows in reference.groupby("trace_id", sort=False):
    trace = traces[trace_id]
    v, w, s, penalty, T = prepare(trace, rows)
    for row in rows.itertuples(index=False):
        for name, scale, width, mask in selected_masks(v, row.R_nominal, w, s, T, row.shape):
            stats = mask_stats(v, mask, T, s, penalty, row.R_nominal, row.eta_lower, row.eta_upper)
            records.append(dict(trace_id=trace_id, shape=row.shape, prompt=row.prompt,
                fraction=row.fraction, N=row.N, R_nominal=row.R_nominal, saturation_rows=s,
                method=name, tile_scale=scale, tile_width=width,
                reference_sha256=reference_sha256, **stats))

## Save the selector results

The reference hash ties this sweep to its bounds. Each row retains the mask intervals, fill, importance, modeled latency and fragmentation measurements, including cases where a fixed-budget gap is undefined.

In [15]:
results = pd.DataFrame(records)
assert not results.duplicated(["trace_id", "fraction", "method"]).any()
results.to_csv(RESULTS, index=False)
results.head()

Out[1]: 
   trace_id    shape  prompt  ...  gap_lower_pct  gap_upper_pct     intervals
0         0  896x896       0  ...            NaN            NaN  [[480, 568]]
1         0  896x896       0  ...   8.881784e-14       0.000091  [[483, 572]]
2         0  896x896       0  ...   8.881784e-14       0.000091  [[483, 572]]
3         0  896x896       0  ...   8.881784e-14       0.000091  [[483, 572]]
4         0  896x896       0  ...   8.881784e-14       0.000091  [[483, 572]]

[5 rows x 24 columns]
